# RentSafe-MY — Scam Classifier Training (ELECTRA Bahasa)

Fine-tunes `mesolitica/electra-base-discriminator-bahasa-cased` into the binary
SCAM / LEGITIMATE classifier for `nlp-service` layer 2, then saves the weights for
download.

Standalone: it trains **only** ELECTRA. Your existing XLM-R model in
`models/xlm-roberta-scam-random` is untouched and is not retrained here. Run this,
then bring the section 7 printout back and the two runs get compared against each
other.

## The settings match your XLM-R run, not the notebook it came from

Read out of `models/xlm-roberta-scam-random/training_args.bin` — the model you
actually trained and kept:

| | value | note |
|---|---|---|
| epochs | **8** | `train_colab.ipynb` says 4; your run used 8 |
| batch size | 16 train / 32 eval | |
| learning rate | 2e-5 | linear schedule, warmup 0.1 |
| weight decay | 0.0 | |
| optimiser | AdamW (torch) | |
| seed | 42 | |
| fp16 | on | |
| max length | 192 | |
| split | `random` | from `FINETUNED_MODEL_DIR` in `.env` |

Everything above is reproduced below so the only difference between the two runs is
the backbone. If you changed something else mid-run last time, change it here too —
otherwise the comparison is measuring the wrong thing.

| | `xlm-roberta-base` (trained) | `electra-base-discriminator-bahasa-cased` (this run) |
|---|---|---|
| Pretraining | 100 languages, CommonCrawl | Malay corpora (Mesolitica / Malaya) |
| Vocab | 250k SentencePiece | 32k WordPiece, cased |
| Parameters | ~278M | ~134M |

Two things about this checkpoint worth knowing:

- It ships as `ElectraForPreTraining`, so loading it for classification prints a
  warning about newly initialised `classifier.*` weights. Expected — the head starts
  from scratch and is what training fits.
- It is **cased**, so nothing should lowercase the text.

**Before running: Runtime → Change runtime type → T4 GPU.**

## 1. Confirm the GPU is attached

In [ ]:
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
else:
    print("No GPU attached.")
    print("Runtime > Change runtime type > T4 GPU, then re-run this cell.")
    print("(Training still works on CPU here, just far slower.)")

## 2. Install dependencies

Pinned to the same `transformers` version as the local service so the saved weights
load without surprises. If Colab asks to restart the session after the install,
restart and re-run from here.

In [ ]:
!pip install -q "transformers==4.46.2" "accelerate>=0.26.0" scikit-learn

## 3. Upload the dataset

Run the cell, then pick **both** `train.csv` and `val.csv` from:

- `rentsafe-my/data/processed/random/` — the split your XLM-R model was trained on

Use `random` if you want the comparison to mean anything; scores from the `grouped`
split are not comparable to a model trained on `random`. If you also trained XLM-R on
`grouped`, run this notebook a second time with those CSVs and `SPLIT_SET` changed to
match.

The held-out real conversations are never uploaded here — they are the final test set
and stay sealed until the backbone is chosen.

In [ ]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)
print("Select train.csv and val.csv from rentsafe-my/data/processed/random/ ...")

for name in files.upload():
    os.replace(name, os.path.join("data", name))

print("Uploaded:", sorted(os.listdir("data")))

## 4. Tokenizer sanity check

Optional — it trains nothing. But this run swaps XLM-R's 250k SentencePiece vocab for
a 32k Malay WordPiece vocab, and that is the one place a backbone change goes wrong
quietly. Heavy `[UNK]` output or text shattered into single characters means stop; the
run would be measuring a broken tokenizer.

The second half checks truncation at `MAX_LENGTH = 192`. The two vocabularies produce
different token counts for the same conversation, so truncation can hit ELECTRA harder
or softer than it hit XLM-R — a real asymmetry hiding inside a "same settings"
comparison. If a large share is truncated, say so when you send the results.

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

MODEL = "mesolitica/electra-base-discriminator-bahasa-cased"
MAX_LENGTH = 192

# This repo ships vocab.txt but no tokenizer.json, so the fast tokenizer is built
# from the vocab on the fly.
check_tokenizer = AutoTokenizer.from_pretrained(MODEL)

sample = "Deposit RM1500 kena bayar dulu sebelum viewing, nanti saya bagi kunci."
pieces = check_tokenizer.tokenize(sample)

print(f"vocab {len(check_tokenizer):,}, fast={check_tokenizer.is_fast}")
print(f"sample -> {len(pieces)} tokens, {pieces.count(check_tokenizer.unk_token)} UNK")
print(pieces)
print()

lengths = np.array([
    len(check_tokenizer.tokenize(text)) for text in pd.read_csv("data/val.csv")["text"]
])
print(f"validation conversations: median {int(np.median(lengths))}, "
      f"p95 {int(np.percentile(lengths, 95))}, max {lengths.max()} tokens")
print(f"truncated at MAX_LENGTH={MAX_LENGTH}: {(lengths > MAX_LENGTH).mean():.1%}")

## 5. Train

Identical to `train_colab.ipynb` apart from `MODEL`, `FINAL_DIR`, and `EPOCHS = 8` to
match what your XLM-R run actually used.

Do not tune these for ELECTRA before the comparison. Tuning one side only is how a
backbone gets chosen for the wrong reason — tune after the decision, not during.

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

CHECKPOINT_DIR = "checkpoints"

# Matches the split your XLM-R model was trained on.
SPLIT_SET = "random"
EPOCHS = 8
SEED = 42
FINAL_DIR = f"electra-bahasa-scam-{SPLIT_SET}"
LABELS = {"LEGITIMATE": 0, "SCAM": 1}
ID2LABEL = {0: "LEGITIMATE", 1: "SCAM"}


def load_frame(path):
    frame = pd.read_csv(path)
    frame["label"] = frame["label"].str.upper().replace({"LEGIT": "LEGITIMATE"})
    invalid = sorted(set(frame["label"]) - set(LABELS))
    if invalid:
        raise ValueError(f"Unsupported labels in {path}: {invalid}")
    return frame


class ScamDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer):
        self.encodings = tokenizer(
            frame["text"].tolist(), truncation=True, max_length=MAX_LENGTH
        )
        self.labels = [LABELS[label] for label in frame["label"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def compute_metrics(prediction):
    predicted = np.argmax(prediction.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        prediction.label_ids, predicted, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(prediction.label_ids, predicted),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


tokenizer = AutoTokenizer.from_pretrained(MODEL)
train_frame = load_frame("data/train.csv")
val_frame = load_frame("data/val.csv")
train_dataset = ScamDataset(train_frame, tokenizer)
val_dataset = ScamDataset(val_frame, tokenizer)

print(f"{len(train_frame)} train / {len(val_frame)} validation conversations")
print("val:", dict(val_frame["label"].value_counts()))

# The "newly initialized: classifier.*" warning below is expected.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABELS,
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Scores, against the baseline that matters

The FYP target is validation F1 >= 0.80, but on this validation set **a model that
answers SCAM every single time scores F1 0.7975** — 63 of the 95 rows are SCAM. The
target is nearly met by a classifier that has learned nothing, so the baseline is
printed alongside your score. ELECTRA has only earned something to the extent it clears
that line, and the per-class table is where you check it is not simply drifting toward
the majority class.

`scam_rate` is the share of validation rows the model called SCAM. The true rate is
0.66; the always-SCAM baseline is 1.00.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

metrics = trainer.evaluate()
prediction = trainer.predict(val_dataset)
predicted = np.argmax(prediction.predictions, axis=1)
actual = prediction.label_ids

# What a classifier that always answers SCAM would score here.
always_scam = np.ones_like(actual)
base_precision, base_recall, base_f1, _ = precision_recall_fscore_support(
    actual, always_scam, average="binary", zero_division=0
)

print(f"model   {MODEL}")
print(f"split   {SPLIT_SET}   epochs {EPOCHS}   seed {SEED}")
print()
print(f"{'':<12} {'accuracy':>9} {'precision':>10} {'recall':>8} {'f1':>8}")
print(f"{'electra':<12} {metrics['eval_accuracy']:>9.4f} {metrics['eval_precision']:>10.4f} "
      f"{metrics['eval_recall']:>8.4f} {metrics['eval_f1']:>8.4f}")
print(f"{'always-SCAM':<12} {accuracy_score(actual, always_scam):>9.4f} "
      f"{base_precision:>10.4f} {base_recall:>8.4f} {base_f1:>8.4f}")
print()
print(f"scam_rate {predicted.mean():.4f}   (true rate {actual.mean():.4f})")
print()

if metrics["eval_f1"] <= base_f1:
    print("Does NOT beat the always-SCAM baseline - this model has not learned")
    print("anything useful yet, whatever the 0.80 target says.")
elif metrics["eval_f1"] >= 0.80:
    print(f"Target met, and {metrics['eval_f1'] - base_f1:.4f} above the trivial baseline.")
else:
    print(f"Below the 0.80 target, {metrics['eval_f1'] - base_f1:+.4f} vs the baseline.")

print()
print(classification_report(
    actual, predicted, target_names=["LEGITIMATE", "SCAM"], digits=4, zero_division=0
))

matrix = confusion_matrix(actual, predicted)
print("                 predicted")
print("                 LEGIT   SCAM")
print(f"actual LEGIT     {matrix[0][0]:>5}  {matrix[0][1]:>5}   <- false alarms")
print(f"actual SCAM      {matrix[1][0]:>5}  {matrix[1][1]:>5}")
print( "                     ^ missed scams")

## 7. Send me this

One block with everything needed to compare against your XLM-R run. Copy the whole
printout.

Send the same numbers for XLM-R if you still have them. If you do not, they can be
recovered without retraining — the saved model in `models/xlm-roberta-scam-random`
can be scored on the same `val.csv` locally in about a minute, since inference fits
in RAM even though training did not.

In [ ]:
import json

report = {
    "model": MODEL,
    "split_set": SPLIT_SET,
    "epochs": EPOCHS,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "train_rows": len(train_frame),
    "val_rows": len(val_frame),
    "accuracy": round(float(metrics["eval_accuracy"]), 4),
    "precision": round(float(metrics["eval_precision"]), 4),
    "recall": round(float(metrics["eval_recall"]), 4),
    "f1": round(float(metrics["eval_f1"]), 4),
    "scam_rate": round(float(predicted.mean()), 4),
    "baseline_f1_always_scam": round(float(base_f1), 4),
    "confusion_matrix": {
        "legit_correct": int(matrix[0][0]),
        "false_alarms": int(matrix[0][1]),
        "missed_scams": int(matrix[1][0]),
        "scam_correct": int(matrix[1][1]),
    },
    "truncated_share": round(float((lengths > MAX_LENGTH).mean()), 4),
    "epoch_history": [
        {k: round(v, 4) for k, v in entry.items() if k in
         ("epoch", "eval_loss", "eval_f1", "eval_accuracy")}
        for entry in trainer.state.log_history if "eval_f1" in entry
    ],
}

print(json.dumps(report, indent=2))

with open(f"electra-results-{SPLIT_SET}.json", "w") as handle:
    json.dump(report, handle, indent=2)

The `epoch_history` above is worth a glance before you send it. With 379 training
conversations and 8 epochs, watch for `eval_f1` peaking early and then sliding while
`eval_loss` climbs — that is overfitting, and it would mean 8 epochs is too many for
this backbone even though it suited XLM-R. `load_best_model_at_end` saves the best
epoch regardless, so the score stands either way; it just changes what you would
recommend in the write-up.

## 8. Save the best weights

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

size = sum(
    os.path.getsize(os.path.join(FINAL_DIR, name)) for name in os.listdir(FINAL_DIR)
)
print(sorted(os.listdir(FINAL_DIR)))
print(f"{size / 1e6:.0f} MB")

## 9. Get the weights off Colab

The export is ~0.5 GB, about half the XLM-R one, so a browser download is usually
fine. Drive is the safer option on a flaky connection. Run **one** of the cells.

In [ ]:
# Option A — download a zip directly
import shutil

archive = shutil.make_archive(FINAL_DIR, "zip", FINAL_DIR)
print(f"{os.path.getsize(archive) / 1e6:.0f} MB")

files.download(archive)
files.download(f"electra-results-{SPLIT_SET}.json")

In [ ]:
# Option B — copy to Google Drive
import shutil

from google.colab import drive

drive.mount("/content/drive")

destination = f"/content/drive/MyDrive/rentsafe-my/{FINAL_DIR}"
os.makedirs(os.path.dirname(destination), exist_ok=True)
if os.path.exists(destination):
    shutil.rmtree(destination)
shutil.copytree(FINAL_DIR, destination)
shutil.copy(f"electra-results-{SPLIT_SET}.json", os.path.dirname(destination))

print("Saved to", destination)

## 10. Put it next to the XLM-R model

Unzip to `nlp-service/models/electra-bahasa-scam-random`, so it contains
`config.json`, `model.safetensors`, `vocab.txt`, and the tokenizer files directly —
not nested inside another folder. Keep `models/xlm-roberta-scam-random` where it is;
both backbones on disk is what makes switching a one-line change once the comparison
is settled.

**Leave `.env` alone for now.** It still points at the XLM-R model, which is correct
until the comparison says otherwise. If ELECTRA wins, the switch is:

```
PRIMARY_MODEL=mesolitica/electra-base-discriminator-bahasa-cased
FINETUNED_MODEL_DIR=./models/electra-bahasa-scam-random
```

## One caveat to carry into the comparison

With 95 validation conversations, one row is worth 1.05 accuracy points, and both
models are single-seed runs. A gap of two or three points between the backbones is a
couple of conversations and a different random seed — not evidence. If the two scores
land close together, the honest reading is that this dataset cannot separate them, and
the choice falls to size, inference cost, and which class each model fails on. That
reading is worth more marks than a confident claim the numbers do not support.